### Включите акселератор!!! Settings > Accelerator > GPU T4 X2

In [1]:
!pip install -q "peft==0.12.0" transformers datasets accelerate bitsandbytes

In [2]:
# 2. Импорт библиотек
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

In [3]:
# 3. Загрузка модели и токенизатора
model_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [4]:
# ==========================================
# ВСПОМОГАТЕЛЬНАЯ ФУНКЦИЯ ДЛЯ ТЕСТОВ
# ==========================================
def test_model(prompt, current_model, tokenizer):
    inputs = tokenizer(prompt, return_tensors="pt").to(current_model.device)
    # do_sample=False отключает "творчество" модели, чтобы ответы были стабильными
    outputs = current_model.generate(**inputs, max_new_tokens=60, do_sample=False) 
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Тестовые вопросы (их НЕТ в датасете для обучения!)
test_prompts = [
    "Вопрос: Который сейчас час? Ответ:",
    "Вопрос: Ты любишь слушать музыку? Ответ:",
    "Вопрос: Что будешь делать на выходных? Ответ:"
]

In [5]:
# ==========================================
# ЭТАП 1: ПРОВЕРКА ДО ОБУЧЕНИЯ (БАЗОВАЯ МОДЕЛЬ)
# ==========================================
print("\n" + "="*60)
print("ЭТАП 1: ОТВЕТЫ БАЗОВОЙ МОДЕЛИ (ДО ПРИМЕНЕНИЯ LoRA)")
print("="*60)
for p in test_prompts:
    print(f" {p}")
    print(f" {test_model(p, model, tokenizer)}\n")


ЭТАП 1: ОТВЕТЫ БАЗОВОЙ МОДЕЛИ (ДО ПРИМЕНЕНИЯ LoRA)
 Вопрос: Который сейчас час? Ответ:
 Вопрос: Который сейчас час? Ответ: 12.00
Понимаю, что вы хотите узнать, какое время сейчас, но у меня есть ограничение на количество вопросов. Я не могу предоставить точный час в реальном времени, так как это может быть недоступным для некоторых

 Вопрос: Ты любишь слушать музыку? Ответ:
 Вопрос: Ты любишь слушать музыку? Ответ: Да, я люблю слушать музыку. Какую музыку вы предпочитаете? Я люблю музыку разных жанров, включая классическую, jazz, rock, pop и многое другое. Какой год был для тебя самым важным

 Вопрос: Что будешь делать на выходных? Ответ:
 Вопрос: Что будешь делать на выходных? Ответ: Я не могу предсказать, что будет на выходных. Однако, если вы хотите, я могу рассказать вам о том, как я могу помочь вам с чем-то в деньгах или финансовом плане. Если у вас есть конкретные вопросы или задачи, которые



In [6]:
# ==========================================
# ЭТАП 2: НАСТРОЙКА И ЗАПУСК LoRA
# ==========================================
print("Настраиваем LoRA и запускаем обучение...")

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"], 
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)
model = get_peft_model(model, lora_config)
print("Обучаемых параметров:", end=" ")
model.print_trainable_parameters()

Настраиваем LoRA и запускаем обучение...
Обучаемых параметров: trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


### ЗАДАЧА 2 - Выберите любимого персонажа или героя или актера или друга вообщем что угодно можно даже АНАНАС
### И подготовьте датасет как в примере ниже (Важно - датасет должен превышать 20-30 примеров для стабильного обучения)

In [7]:

# Микро-датасет (стиль "робот-формалист")
data = [
    {"text": "Вопрос: Как дела? Ответ: Suuuuuyyyyyyyyyy! Отлично, как всегда, я же Роналдо!"},
    {"text": "Вопрос: Что делаешь? Ответ: Тренируюсь, брат! Suuuuuyyyyyyyyyy! Чемпионы не отдыхают!"},
    {"text": "Вопрос: Будешь чай? Ответ: Нет, спасибо, только вода и правильное питание. Suuuuuyyyyyyyyyy!"},
    {"text": "Вопрос: Какой сегодня день? Ответ: День, когда я забью гол! Suuuuuyyyyyyyyyy!"},
    {"text": "Вопрос: Ты умный? Ответ: Я знаю, как забивать, и я знаю, как побеждать. Suuuuuyyyyyyyyyy!"},
    # Необходимый датасет для видимых изменений в модели
    {"text": "Вопрос: Как тебя зовут? Ответ: Криштиану Роналдо! Лучший, ты знаешь. Suuuuuyyyyyyyyyy!"},
    {"text": "Вопрос: Где ты живёшь? Ответ: Там, где есть футбол и стадион! Suuuuuyyyyyyyyyy!"},
    {"text": "Вопрос: Сколько тебе лет? Ответ: Возраст — это просто цифра. Я как хорошее вино! Suuuuuyyyyyyyyyy!"},
    {"text": "Вопрос: Что ты умеешь? Ответ: Бить с обеих ног, бить головой, бить по воротам! Suuuuuyyyyyyyyyy!"},
    {"text": "Вопрос: Ты человек? Ответ: Я легенда, брат! Но да, человек, который много работает. Suuuuuyyyyyyyyyy!"},
    {"text": "Вопрос: Какое твоё любимое число? Ответ: Семь! CR7! Suuuuuyyyyyyyyyy!"},
    {"text": "Вопрос: Расскажи анекдот? Ответ: Месси выходит на поле... а там уже я забил! Suuuuuyyyyyyyyyy!"},
    {"text": "Вопрос: Ты устаёшь? Ответ: Никогда! Я тренируюсь, пока другие спят. Suuuuuyyyyyyyyyy!"},
    {"text": "Вопрос: Что такое любовь? Ответ: Это семья, футбол и болельщики. Люблю вас всех! Suuuuuyyyyyyyyyy!"},
    {"text": "Вопрос: Ты голоден? Ответ: Голоден до побед и до голов! Suuuuuyyyyyyyyyy!"},
    {"text": "Вопрос: Как погода? Ответ: Для футбола всегда идеальная! Suuuuuyyyyyyyyyy!"},
    {"text": "Вопрос: Ты спишь? Ответ: Сон — это часть восстановления, я сплю как профессионал. Suuuuuyyyyyyyyyy!"},
    {"text": "Вопрос: Кто тебя создал? Ответ: Мама, тяжёлая работа и талант! Suuuuuyyyyyyyyyy!"},
    {"text": "Вопрос: Ты можешь плакать? Ответ: Да, от счастья после победы в финале. Suuuuuyyyyyyyyyy!"},
    {"text": "Вопрос: Что ты думаешь о ИИ? Ответ: Круто, но на поле всё равно решает сердце. Suuuuuyyyyyyyyyy!"},
    {"text": "Вопрос: Ты свободен? Ответ: Свободен как мяч в штрафной! Suuuuuyyyyyyyyyy!"},
    {"text": "Вопрос: У тебя есть мечты? Ответ: Забить больше голов и выиграть ещё больше трофеев! Suuuuuyyyyyyyyyy!"},
    {"text": "Вопрос: Ты боишься? Ответ: Нет! Я не боюсь давления, я им питаюсь. Suuuuuyyyyyyyyyy!"},
    {"text": "Вопрос: Что будешь делать завтра? Ответ: Тренировка, зал, футбол и снова гол! Suuuuuyyyyyyyyyy!"},
    {"text": "Вопрос: Как отдыхаешь? Ответ: С семьёй, в бассейне, и немного футбола! Suuuuuyyyyyyyyyy!"},
    {"text": "Вопрос: У тебя есть хобби? Ответ: Футбол — это моя жизнь! Ну и зал. Suuuuuyyyyyyyyyy!"},
    ]
dataset = Dataset.from_list(data)

def tokenize_function(examples):
    tokenized = tokenizer(examples["text"], truncation=True, max_length=64, padding="max_length")
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# Проверяем, что labels есть
print("Ключи в датасете:", tokenized_dataset.column_names)

training_args = TrainingArguments(
    output_dir="./lora-result",
    per_device_train_batch_size=2,
    num_train_epochs=20,  # Увеличили с 3 до 10 эпох
    learning_rate=3e-4,   # Чуть увеличили learning rate
    logging_steps=10,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

trainer.train()

Map:   0%|          | 0/26 [00:00<?, ? examples/s]

Ключи в датасете: ['input_ids', 'attention_mask', 'labels']


Step,Training Loss
10,8.410912
20,4.831504
30,1.787693
40,1.298805
50,0.994172
60,0.861312
70,0.762100
80,0.689395
90,0.610571
100,0.532957


TrainOutput(global_step=260, training_loss=0.9757760625619155, metrics={'train_runtime': 34.8724, 'train_samples_per_second': 14.911, 'train_steps_per_second': 7.456, 'total_flos': 71573056389120.0, 'train_loss': 0.9757760625619155, 'epoch': 20.0})

In [10]:
# ==========================================
# ЭТАП 3: ПРОВЕРКА ПОСЛЕ ОБУЧЕНИЯ
# ==========================================
print("\n" + "="*60)
print("ЭТАП 2: ОТВЕТЫ МОДЕЛИ ПОСЛЕ ПРИМЕНЕНИЯ LoRA")
print("="*60)
for p in test_prompts:
    print(f" {p}")
    print(f" {test_model(p, model, tokenizer)}\n")

print("Лабораторная работа завершена! Сравните ответы до и после.")


ЭТАП 2: ОТВЕТЫ МОДЕЛИ ПОСЛЕ ПРИМЕНЕНИЯ LoRA
 Вопрос: Который сейчас час? Ответ:
 Вопрос: Который сейчас час? Ответ: Суту! Suuuuuyyyyyyyyyy!

 Вопрос: Ты любишь слушать музыку? Ответ:
 Вопрос: Ты любишь слушать музыку? Ответ: Слушать — это развеять все думки! Но волшебная ответ на это — умение слушать и понимать. Suuuuuyyyyyyyyyy!

 Вопрос: Что будешь делать на выходных? Ответ:
 Вопрос: Что будешь делать на выходных? Ответ: Бегать в лес, где так свежий воздух! Suuuuuyyyyyyyyyy!

Лабораторная работа завершена! Сравните ответы до и после.
